In [3]:
import os
import shutil
import hashlib
import math
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from PIL import Image, UnidentifiedImageError
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# =========================
# CONFIG
# =========================
RAW_DATASET_DIR   = Path("dataset/wikiart")
CLEAN_DATASET_DIR = Path("dataset/wikiart_clean")
SPLIT_DATASET_DIR = Path("dataset/wikiart_split")

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MIN_WIDTH  = 64
MIN_HEIGHT = 64

TRAIN_SIZE   = 0.70
VAL_SIZE     = 0.15
TEST_SIZE    = 0.15
RANDOM_STATE = 42

assert abs(TRAIN_SIZE + VAL_SIZE + TEST_SIZE - 1.0) < 1e-6, "Os splits tem de somar 1."


# Vision Transformer (ViT) - From Scratch
Custom implementation of a Vision Transformer for painting classification (WikiArt).


## Data Cleaning and Organisation

In [5]:
def is_valid_image_file(file_path):
    return file_path.suffix.lower() in IMG_EXTENSIONS


def clean_dataset(raw_dir, clean_dir, min_width=64, min_height=64):
    clean_dir.mkdir(parents=True, exist_ok=True)
    removed_files, kept_files = [], []

    class_dirs = [d for d in raw_dir.iterdir() if d.is_dir()]

    for class_dir in class_dirs:
        target_class_dir = clean_dir / class_dir.name
        target_class_dir.mkdir(parents=True, exist_ok=True)

        for file_path in class_dir.iterdir():
            if not file_path.is_file():
                continue
            if not is_valid_image_file(file_path):
                removed_files.append((str(file_path), "invalid"))
                continue
            try:
                with Image.open(file_path) as img:
                    img.verify()
                with Image.open(file_path) as img:
                    img = img.convert("RGB")
                    if img.width < min_width or img.height < min_height:
                        removed_files.append((str(file_path), f"image too small: {img.width}x{img.height}"))
                        continue
                    output_filename = file_path.stem + ".jpg"
                    output_path = target_class_dir / output_filename
                    img.save(output_path, format="JPEG", quality=95)
                    kept_files.append((str(output_path), class_dir.name, img.width, img.height))
            except (UnidentifiedImageError, OSError, Image.DecompressionBombError) as e:
                removed_files.append((str(file_path), f"error opening/processing: {e}"))

    kept_df    = pd.DataFrame(kept_files,    columns=["filepath", "label", "width", "height"])
    removed_df = pd.DataFrame(removed_files, columns=["filepath", "reason"])
    return kept_df, removed_df


kept_df, removed_df = clean_dataset(
    raw_dir=RAW_DATASET_DIR,
    clean_dir=CLEAN_DATASET_DIR,
    min_width=MIN_WIDTH,
    min_height=MIN_HEIGHT
)

print("Valid:",   len(kept_df))
print("Removed:", len(removed_df))

class_counts = kept_df["label"].value_counts().sort_values(ascending=False)
print("\nNumber of images per author:")
print(class_counts)


Valid: 13340
Removed: 0

Number of images per author:
label
Vincent_van_Gogh         1322
Nicholas_Roerich         1274
Pierre_Auguste_Renoir     975
Claude_Monet              934
Pyotr_Konchalovsky        644
Camille_Pissarro          621
Albrecht_Durer            580
John_Singer_Sargent       549
Rembrandt                 544
Marc_Chagall              536
Pablo_Picasso             534
Gustave_Dore              528
Boris_Kustodiev           444
Edgar_Degas               428
Paul_Cezanne              406
Ivan_Aivazovsky           404
Martiros_Saryan           403
Eugene_Boudin             389
Childe_Hassam             385
Ilya_Repin                378
Ivan_Shishkin             364
Raphael_Kirchner          362
Salvador_Dali             336
Name: count, dtype: int64


## Outlier Detection
Before splitting, we detect and remove three types of problematic images:
1. **Exact duplicates** (MD5 hash) - prevents data leakage across splits
2. **Extreme aspect ratios** (>5:1 or <1:5) - heavily distorted when resized to 224x224
3. **Near-black or near-white images** - little to no visual information


In [6]:
# 1. Exact duplicates by MD5
def compute_md5(filepath):
    with open(filepath, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

kept_df["md5"] = kept_df["filepath"].apply(compute_md5)
duplicates_mask = kept_df.duplicated(subset="md5", keep="first")
print(f"Duplicates found: {duplicates_mask.sum()}")
kept_df = kept_df[~duplicates_mask].reset_index(drop=True)

# 2. Extreme aspect ratio (> 5:1 or < 1:5)
kept_df["aspect_ratio"] = kept_df["width"] / kept_df["height"]
extreme_ratio_mask = (kept_df["aspect_ratio"] > 5) | (kept_df["aspect_ratio"] < 0.2)
print(f"Extreme aspect ratios: {extreme_ratio_mask.sum()}")
kept_df = kept_df[~extreme_ratio_mask].reset_index(drop=True)

# 3. Near-black or near-white images
def is_low_information(filepath, black_thresh=15, white_thresh=240, ratio_thresh=0.95):
    with Image.open(filepath) as img:
        arr = np.array(img.convert("RGB"))
    n_total  = arr.shape[0] * arr.shape[1]
    n_dark   = np.sum(np.all(arr < black_thresh,  axis=2))
    n_bright = np.sum(np.all(arr > white_thresh,  axis=2))
    return (n_dark / n_total > ratio_thresh) or (n_bright / n_total > ratio_thresh)

low_info_mask = kept_df["filepath"].apply(is_low_information)
print(f"Low information images: {low_info_mask.sum()}")
kept_df = kept_df[~low_info_mask].reset_index(drop=True)
print(f"\nTotal after outlier detection: {len(kept_df)}")
print(kept_df["label"].value_counts())


Duplicates found: 2
Extreme aspect ratios: 0
Low information images: 0

Total after outlier detection: 13338
label
Vincent_van_Gogh         1322
Nicholas_Roerich         1274
Pierre_Auguste_Renoir     975
Claude_Monet              934
Pyotr_Konchalovsky        644
Camille_Pissarro          621
Albrecht_Durer            580
John_Singer_Sargent       549
Rembrandt                 544
Marc_Chagall              536
Pablo_Picasso             534
Gustave_Dore              528
Boris_Kustodiev           444
Edgar_Degas               428
Paul_Cezanne              406
Ivan_Aivazovsky           404
Martiros_Saryan           403
Eugene_Boudin             389
Childe_Hassam             383
Ilya_Repin                378
Ivan_Shishkin             364
Raphael_Kirchner          362
Salvador_Dali             336
Name: count, dtype: int64


## Train/Val/Test

In [7]:
def create_splits(df, train_size=0.70, val_size=0.15, test_size=0.15, random_state=42):
    train_df, temp_df = train_test_split(
        df, test_size=(1 - train_size),
        stratify=df["label"], random_state=random_state
    )
    val_relative = val_size / (val_size + test_size)
    val_df, test_df = train_test_split(
        temp_df, test_size=(1 - val_relative),
        stratify=temp_df["label"], random_state=random_state
    )
    return train_df, val_df, test_df


train_df, val_df, test_df = create_splits(
    kept_df, TRAIN_SIZE, VAL_SIZE, TEST_SIZE, RANDOM_STATE
)

print("Train:",      len(train_df))
print("Validation:", len(val_df))
print("Test:",       len(test_df))


def copy_files(df, split_name, base_dir):
    for _, row in df.iterrows():
        dst_dir = os.path.join(base_dir, split_name, row["label"])
        os.makedirs(dst_dir, exist_ok=True)
        dst = os.path.join(dst_dir, os.path.basename(row["filepath"]))
        if not os.path.exists(dst):
            shutil.copy(row["filepath"], dst)

copy_files(train_df, "train", SPLIT_DATASET_DIR)
copy_files(val_df,   "val",   SPLIT_DATASET_DIR)
copy_files(test_df,  "test",  SPLIT_DATASET_DIR)


Train: 9336
Validation: 2001
Test: 2001


## Datasets for ViT
The ViT normalises to `[-1, 1]` (divide by 127.5, subtract 1).

`label_mode="categorical"` (one-hot) is required for **MixUp** and **Label Smoothing**.

`IMG_SIZE = 224"` for compatibility with *ImageNet*.

`BATCH_SIZE = 32"` for larger datasets a larger batch size, such as 128 or 256, can be more efficient and allow for faster training, however to not compromise GPUs and VRAM memory we decided to opt for a smaller number.


In [8]:
IMG_SIZE   = 224
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# label_mode='categorical' -> one-hot labels (required for MixUp + Label Smoothing)
train_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    label_mode="categorical"
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "val",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="categorical"
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "test",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="categorical"
)

class_names = train_ds.class_names
num_classes  = len(class_names)
print(f"Classes ({num_classes}): {class_names}")


Found 9336 files belonging to 23 classes.
Found 2001 files belonging to 23 classes.
Found 2001 files belonging to 23 classes.
Classes (23): ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']


## Unbalanced DataSet Solutions

* **Class Weights:** Assigns a higher penalty to mistakes made on the minority class during training, forcing the model to prioritize learning its patterns without changing the underlying data.
* **Augmentation:** Increases the size and diversity of the minority class by creating synthetic variations of existing samples (e.g., rotating images) to balance the distribution.

In [9]:
labels_list    = train_df["label"].values
unique_classes = np.unique(labels_list)

weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=unique_classes,
    y=labels_list
)
class_weights = {i: w for i, w in enumerate(weights)}
print("Weights:", {class_names[i]: round(class_weights[i], 2) for i in range(num_classes)})


Weights: {'Albrecht_Durer': 1.0, 'Boris_Kustodiev': 1.31, 'Camille_Pissarro': 0.93, 'Childe_Hassam': 1.51, 'Claude_Monet': 0.62, 'Edgar_Degas': 1.35, 'Eugene_Boudin': 1.49, 'Gustave_Dore': 1.1, 'Ilya_Repin': 1.53, 'Ivan_Aivazovsky': 1.43, 'Ivan_Shishkin': 1.59, 'John_Singer_Sargent': 1.06, 'Marc_Chagall': 1.08, 'Martiros_Saryan': 1.44, 'Nicholas_Roerich': 0.46, 'Pablo_Picasso': 1.09, 'Paul_Cezanne': 1.43, 'Pierre_Auguste_Renoir': 0.6, 'Pyotr_Konchalovsky': 0.9, 'Raphael_Kirchner': 1.6, 'Rembrandt': 1.07, 'Salvador_Dali': 1.73, 'Vincent_van_Gogh': 0.44}


Applied only during training. `RandomRotation` is kept low (0.05) because paintings are sensitive to strong rotations.


In [10]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
    tf.keras.layers.RandomBrightness(0.1),
], name="augmentation")


## MixUp
This function implements the **Mixup technique**, a form of data augmentation that creates convex combinations of pairs of examples and their labels.

- Linear Interpolation: Blends two random images and their corresponding labels using a ratio ($\lambda$) sampled from a Beta distribution.
- Batch Shuffling: Randomly pairs each image in the current batch with another image from the same batch to create unique combinations.
- Label Smoothing Effect: Instead of "hard" one-hot labels, the model learns on "soft" mixed labels, which improves generalization and robustness against adversarial examples.
- Mathematical Principle: For a pair of inputs $(x_i, y_i)$ and $(x_j, y_j)$, it generates a synthetic sample:
  $$\begin{aligned} 
\hat{x} &= \lambda x_i + (1 - \lambda) x_j \\ 
\hat{y} &= \lambda y_i + (1 - \lambda) y_j 
\end{aligned}$$


<span style="color:red">Requires One-Hot encoding!</span>

In [11]:
def mixup(images, labels, alpha=0.2):
    """Mix two batches of images and labels. lam >= 0.5 so first image stays dominant."""
    batch_size = tf.shape(images)[0]
    lam        = tf.random.uniform(shape=[], minval=0.0, maxval=1.0)
    lam        = tf.maximum(lam, 1.0 - lam)

    indices = tf.random.shuffle(tf.range(batch_size))
    images2 = tf.gather(images, indices)
    labels2 = tf.gather(labels, indices)

    mixed_images = lam * images + (1.0 - lam) * images2
    mixed_labels = lam * labels + (1.0 - lam) * labels2
    return mixed_images, mixed_labels


def normalize_vit(image, label):
    """Normalise pixels to [-1, 1], as expected by the ViT."""
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    return image, label


def preprocess_train(image, label):
    # 1. Augmentation (pixel space [0, 255])
    image = data_augmentation(image, training=True)
    # 2. Normalise to [-1, 1]
    image, label = normalize_vit(image, label)
    # 3. MixUp
    image, label = mixup(image, label)
    return image, label


def preprocess_eval(image, label):
    return normalize_vit(image, label)


train_ds_vit = train_ds.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds_vit   = val_ds.map(preprocess_eval,    num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds_vit  = test_ds.map(preprocess_eval,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print("ViT datasets ready.")


ViT datasets ready.


## 2. ViT Hyperparameters

| Parameter | Value | Justification |
|---|---|---|
| `PATCH_SIZE` | 16 | Divides 224x224 into 14x14 = 196 patches |
| `PROJECTION_DIM` | 128 | Embedding dimension per patch |
| `NUM_HEADS` | 4 | Multi-head attention |
| `TRANSFORMER_LAYERS` | 6 | Encoder depth |
| `MLP_HEAD_UNITS` | [256, 128] | Final classification MLP |
| `DROPOUT` | 0.1 | Regularisation |


In [12]:
PATCH_SIZE         = 16
NUM_PATCHES        = (IMG_SIZE // PATCH_SIZE) ** 2   # 196
PROJECTION_DIM     = 128
NUM_HEADS          = 4
TRANSFORMER_LAYERS = 6
MLP_HEAD_UNITS     = [256, 128]
DROPOUT            = 0.1
EPOCHS             = 50
LEARNING_RATE      = 1e-3
WEIGHT_DECAY       = 1e-4

print(f"Patches per image: {NUM_PATCHES}")
print(f"Each patch: {PATCH_SIZE}x{PATCH_SIZE}x3 = {PATCH_SIZE*PATCH_SIZE*3} values")


Patches per image: 196
Each patch: 16x16x3 = 768 values


## 3. ViT Building Blocks

### 3.1 Patch Embedding
Divides the image in patches and projects in `PROJECTION_DIM` dimensions.


In [13]:
class PatchEmbedding(tf.keras.layers.Layer):
    """
    Divide the image into non-overlapping patches and project each patch
    into a vector of dimension PROJECTION_DIM.
    Uses Conv2D with kernel and stride = PATCH_SIZE.
    """
    def __init__(self, patch_size, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.patch_size     = patch_size
        self.projection_dim = projection_dim
        self.projection = tf.keras.layers.Conv2D(
            filters     = projection_dim,
            kernel_size = patch_size,
            strides     = patch_size,
            padding     = "valid"
        )
        self.flatten = tf.keras.layers.Reshape((-1, projection_dim))

    def call(self, x):
        x = self.projection(x)
        x = self.flatten(x)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size, "projection_dim": self.projection_dim})
        return config


- Input: The model receives a 2D image (e.g., 224x224 pixels).
- Projection (Conv2D): Groups pixels into patches. With patch_size=16, the image is a 14x14 grid.
- Transformation: Each 16x16 patch becomes a single embedding vector.
- The Sequence (Reshape): Organizes the 196 patches (14*14).


### 3.2 Positional Embedding
Adds learnable position embeddings + the special `[CLS]` token that represents the entire image.


In [14]:
class PositionalEmbedding(tf.keras.layers.Layer):
    """
    Concatenate a learnable [CLS] token to the sequence of patches,
    and add learnable positional embeddings.
    Final sequence: [CLS, patch_1, ..., patch_N] with N+1 positions.
    """
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            name="cls_token", shape=(1, 1, self.projection_dim),
            initializer="zeros", trainable=True
        )
        self.pos_embedding = self.add_weight(
            name="pos_embedding", shape=(1, self.num_patches + 1, self.projection_dim),
            initializer="random_normal", trainable=True
        )

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls_tokens = tf.broadcast_to(self.cls_token, [batch_size, 1, self.projection_dim])
        x = tf.concat([cls_tokens, x], axis=1)
        x = x + self.pos_embedding
        return x

    def get_config(self):
        config = super().get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config


We create a set of coordinates so the model knows where each patch belongs in the original picture.


### 3.3 Transformer Encoder Block
**LayerNorm -> Multi-Head Attention -> residual** + **LayerNorm -> MLP -> residual**.


In [15]:
class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, projection_dim, num_heads, dropout, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.attn  = tf.keras.layers.MultiHeadAttention(
            num_heads = num_heads,
            key_dim   = projection_dim // num_heads,
            dropout   = dropout
        )
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.mlp   = tf.keras.Sequential([
            tf.keras.layers.Dense(projection_dim * 4, activation="gelu"),
            tf.keras.layers.Dropout(dropout),
            tf.keras.layers.Dense(projection_dim),
            tf.keras.layers.Dropout(dropout),
        ])
        self.drop1 = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        # Block 1: Multi-Head Self-Attention + residual
        x_norm   = self.norm1(x)
        attn_out = self.attn(x_norm, x_norm, training=training)
        x        = x + self.drop1(attn_out, training=training)
        # Block 2: MLP Feed-Forward + residual
        x_norm = self.norm2(x)
        x      = x + self.mlp(x_norm, training=training)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "projection_dim" : self.attn.key_dim * self.attn.num_heads,
            "num_heads"      : self.attn.num_heads,
            "dropout"        : self.drop1.rate
        })
        return config


- **Multi-Head Self-Attention** allows every patch to communicate with all others.
- **MLP (Feed-Forward)** independently processes information gathered by each patch.


## 4. Full ViT Model

Full pipeline:
```
Image (224x224x3)
  -> PatchEmbedding      -> (196, 128)
  -> PositionalEmbedding -> (197, 128)   # +CLS
  -> Dropout
  -> 6x TransformerEncoderBlock
  -> LayerNorm
  -> CLS token [:, 0, :]  -> (128,)
  -> MLP Head -> num_classes
```


In [16]:
def build_vit(
    img_size           = IMG_SIZE,
    patch_size         = PATCH_SIZE,
    num_patches        = NUM_PATCHES,
    projection_dim     = PROJECTION_DIM,
    num_heads          = NUM_HEADS,
    transformer_layers = TRANSFORMER_LAYERS,
    mlp_head_units     = MLP_HEAD_UNITS,
    dropout            = DROPOUT,
    num_classes        = num_classes
):
    inputs = tf.keras.Input(shape=(img_size, img_size, 3))

    # 1. Patch + Positional Embedding
    x = PatchEmbedding(patch_size, projection_dim, name="patch_embedding")(inputs)
    x = PositionalEmbedding(num_patches, projection_dim, name="pos_embedding")(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    # 2. Transformer Encoder
    for i in range(transformer_layers):
        x = TransformerEncoderBlock(
            projection_dim, num_heads, dropout,
            name=f"transformer_block_{i}"
        )(x)

    # 3. Final Layer Norm
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="ln_final")(x)

    # 4. Extract [CLS] token
    x = x[:, 0, :]   # (B, D)

    # 5. Classification MLP head
    for units in mlp_head_units:
        x = tf.keras.layers.Dense(units, activation="gelu")(x)
        x = tf.keras.layers.Dropout(dropout)(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="classifier")(x)

    return tf.keras.Model(inputs, outputs, name="ViT_from_scratch")


vit_model = build_vit()
vit_model.summary(expand_nested=False)


Model: "ViT_from_scratch"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding                 │ (None, 196, 128)       │        98,432 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pos_embedding                   │ (None, 197, 128)       │        25,344 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 197, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_0             │ (None, 197, 128)       │       198,272 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 197, 128)       │       198,272 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 197, 128)       │       198,272 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 197, 128)       │       198,272 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ (None, 197, 128)       │       198,272 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ (None, 197, 128)       │       198,272 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ln_final (LayerNormalization)   │ (None, 197, 128)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 23)             │         2,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,382,551 (5.27 MB)

 Trainable params: 1,382,551 (5.27 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Compilation and Callbacks

- **Label Smoothing** (`CategoricalCrossentropy(label_smoothing=0.1)`) - prevents overconfident predictions, especially useful for visually similar artists.
- **Cosine Decay + Warmup** - replaces `ReduceLROnPlateau` with a smooth schedule. 5 epochs of warmup avoid large gradient steps at the start.
- **F1Score metric** - built-in Keras macro F1.`threshold=None` uses argmax internally, which is used for softmax outputs.


In [18]:
# Label Smoothing
# Prevents overconfident predictions.
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

# Cosine Decay + Warmup
# Replaces ReduceLROnPlateau with a smooth cosine schedule.
steps_per_epoch = len(train_df) // BATCH_SIZE
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = steps_per_epoch * 5   # 5 epochs of warmup

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate = LEARNING_RATE,
    decay_steps           = total_steps,
    warmup_target         = LEARNING_RATE,
    warmup_steps          = warmup_steps,
    alpha                 = 1e-7
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate = lr_schedule,
    weight_decay  = WEIGHT_DECAY
)

# With one-hot labels the metric name is 'f1_score' -> monitored as 'val_f1_score'
f1_metric = tf.keras.metrics.F1Score(
    average     = "macro",
    threshold   = None,
    name        = "f1_score"
)

vit_model.compile(
    optimizer = optimizer,
    loss      = loss_fn,
    metrics   = [f1_metric]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor              = "val_f1_score",
        patience             = 10,
        mode                 = "max",
        restore_best_weights = True,
        verbose              = 1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath       = "vit_best.keras",
        monitor        = "val_f1_score",
        mode           = "max",
        save_best_only = True,
        verbose        = 1
    )
]

print("Model compiled.")
print(f"Monitoring: val_f1_score | LR: CosineDecay with {warmup_steps} warmup steps")


Model compiled.
Monitoring: val_f1_score | LR: CosineDecay with 1455 warmup steps


## 6. Training


In [19]:
history = vit_model.fit(
    train_ds_vit,
    validation_data = val_ds_vit,
    epochs          = EPOCHS,
    class_weight    = class_weights,
    callbacks       = callbacks
)


Epoch 1/50
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - f1_score: 0.0579 - loss: 3.1237
Epoch 1: val_f1_score improved from None to 0.10085, saving model to vit_best.keras

Epoch 1: finished saving model to vit_best.keras
292/292 ━━━━━━━━━━━━━━━━━━━━ 508s 2s/step - f1_score: 0.0741 - loss: 3.0809 - val_f1_score: 0.1008 - val_loss: 2.9000
Epoch 2/50
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - f1_score: 0.0878 - loss: 3.0005
Epoch 2: val_f1_score did not improve from 0.10085
292/292 ━━━━━━━━━━━━━━━━━━━━ 475s 2s/step - f1_score: 0.0980 - loss: 2.9874 - val_f1_score: 0.0925 - val_loss: 2.8847
Epoch 3/50
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - f1_score: 0.0985 - loss: 2.9578
Epoch 3: val_f1_score improved from 0.10085 to 0.11376, saving model to vit_best.keras

Epoch 3: finished saving model to vit_best.keras
292/292 ━━━━━━━━━━━━━━━━━━━━ 478s 2s/step - f1_score: 0.1082 - loss: 2.9650 - val_f1_score: 0.1138 - val_loss: 2.8042
Epoch 4/50
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - f1_score: 0.1084

KeyboardInterrupt: 

## 7. Training Curves


In [ ]:
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("ViT - Training History", fontsize=14, fontweight="bold")

    ax = axes[0]
    ax.plot(history.history["loss"],     label="Train Loss",      color="steelblue")
    ax.plot(history.history["val_loss"], label="Validation Loss", color="tomato", linestyle="--")
    ax.set_title("Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(history.history["f1_score"],     label="Train F1",      color="steelblue")
    ax.plot(history.history["val_f1_score"], label="Validation F1", color="tomato", linestyle="--")
    ax.set_title("F1 Macro")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("F1")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("vit_training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_training_history(history)


## 8. Evaluation on the Test Set


In [ ]:
results       = vit_model.evaluate(test_ds_vit, verbose=1)
test_loss     = results[0]
test_f1_score = results[1]

print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Test F1 Macro : {test_f1_score:.4f} ({test_f1_score*100:.2f}%)")


## 9. Predictions and Detailed Metrics

### Improvement 4 - TTA (Test Time Augmentation)
Runs each test image 5 times with different augmentations and averages the probabilities before deciding the class. Improves F1 without retraining.


In [ ]:
def predict_tta(model, dataset, n_aug=5):
    """
    For each batch: 1 clean pass + (n_aug - 1) augmented passes.
    Averages softmax probabilities, then argmax.
    """
    y_true, y_pred = [], []

    for images, labels in dataset:
        # images are already normalised to [-1, 1]
        preds = tf.cast(model(images, training=False), tf.float32)

        for _ in range(n_aug - 1):
            # Invert normalisation -> augment -> re-normalise
            imgs_255 = (images + 1.0) * 127.5
            imgs_aug = data_augmentation(imgs_255, training=True)
            imgs_aug = tf.cast(imgs_aug, tf.float32) / 127.5 - 1.0
            preds   += tf.cast(model(imgs_aug, training=False), tf.float32)

        y_true.extend(tf.argmax(labels, axis=1).numpy())
        y_pred.extend(tf.argmax(preds,  axis=1).numpy())

    return y_true, y_pred


y_true, y_pred = predict_tta(vit_model, test_ds_vit, n_aug=5)
print(f"Predictions: {len(y_pred)} samples")


### 9.1 Classification Report


In [ ]:
report = classification_report(
    y_true, y_pred,
    target_names = class_names,
    digits       = 4
)
print("Classification Report:")
print(report)


### 9.2 Confusion Matrix


In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, normalize=True):
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm_plot = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        fmt, title = ".2f", "Confusion Matrix (normalised)"
    else:
        cm_plot = cm
        fmt, title = "d", "Confusion Matrix (counts)"

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm_plot, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        ax=ax, linewidths=0.5
    )
    ax.set_title(f"ViT - {title}", fontsize=13, fontweight="bold", pad=15)
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label",      fontsize=11)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig("vit_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_confusion_matrix(y_true, y_pred, class_names, normalize=True)


### 9.3 Per-Class F1 Score


In [ ]:
def plot_per_class_f1(y_true, y_pred, class_names):
    f1_values = f1_score(y_true, y_pred, average=None)
    f1_mean   = np.mean(f1_values)

    colors = ["#2ecc71" if f >= 0.7 else "#e67e22" if f >= 0.4 else "#e74c3c"
              for f in f1_values]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(class_names, f1_values * 100, color=colors, edgecolor="white", linewidth=0.8)

    for bar, f1 in zip(bars, f1_values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{f1*100:.1f}%",
            ha="center", va="bottom", fontsize=9, fontweight="bold"
        )

    ax.set_ylim(0, 115)
    ax.set_title("ViT - F1-Score per Class", fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Class",        fontsize=11)
    ax.set_ylabel("F1-Score (%)", fontsize=11)

    mean_line = ax.axhline(y=f1_mean * 100, color="steelblue", linestyle="--",
                           alpha=0.8, label=f"Mean: {f1_mean*100:.1f}%")
    legend_patches = [
        mpatches.Patch(color="#2ecc71", label="Good (>= 70%)"),
        mpatches.Patch(color="#e67e22", label="Fair (40-70%)"),
        mpatches.Patch(color="#e74c3c", label="Poor (< 40%)"),
        mean_line
    ]
    ax.legend(handles=legend_patches, loc="upper right", frameon=True)
    ax.grid(axis="y", linestyle=":", alpha=0.6)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig("vit_f1_per_class.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\nDetailed F1-Score per class:")
    for name, f1 in zip(class_names, f1_values):
        print(f"  {name:<25}: {f1*100:.2f}%")

plot_per_class_f1(y_true, y_pred, class_names)


### 9.4 Prediction Examples


In [ ]:
def plot_predictions(test_ds_raw, vit_model, class_names, n=12):
    """
    Shows images from the test set with prediction and true label.
    Green = correct, Red = wrong.
    Uses original test_ds (without ViT normalization) for visualization.
    """
    images, labels = [], []
    for img_batch, lbl_batch in test_ds_raw:
        images.append(img_batch.numpy())
        labels.append(np.argmax(lbl_batch.numpy(), axis=1))  # one-hot -> int
        if sum(len(l) for l in labels) >= n:
            break

    images = np.concatenate(images, axis=0)[:n]
    labels = np.concatenate(labels, axis=0)[:n]

    imgs_norm   = images.astype("float32") / 127.5 - 1.0
    preds_probs = vit_model.predict(imgs_norm, verbose=0)
    preds       = np.argmax(preds_probs, axis=1)

    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    fig.suptitle("ViT - Prediction Examples", fontsize=13, fontweight="bold")

    for i, ax in enumerate(axes.flat):
        if i >= n:
            ax.axis("off")
            continue
        ax.imshow(images[i].astype("uint8"))
        correct = preds[i] == labels[i]
        color   = "green" if correct else "red"
        ax.set_title(
            f"True: {class_names[labels[i]]}\nPred: {class_names[preds[i]]} ({preds_probs[i, preds[i]]*100:.0f}%)",
            color=color, fontsize=8
        )
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("vit_prediction_examples.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_predictions(test_ds, vit_model, class_names, n=12)


## 10. Final Summary


In [ ]:
report_dict = classification_report(
    y_true, y_pred,
    target_names = class_names,
    output_dict  = True
)
summary = pd.DataFrame(report_dict).T.round(4)

print("=" * 55)
print("  ViT from Scratch - Metrics Summary")
print("=" * 55)
print(f"  Model parameters     : {vit_model.count_params():,}")
print(f"  Patch size           : {PATCH_SIZE}x{PATCH_SIZE}")
print(f"  Num patches          : {NUM_PATCHES}")
print(f"  Projection dim       : {PROJECTION_DIM}")
print(f"  Transformer layers   : {TRANSFORMER_LAYERS}")
print(f"  Num heads            : {NUM_HEADS}")
print("-" * 55)
print(f"  Test Loss            : {test_loss:.4f}")
print(f"  Test F1 Macro        : {test_f1_score*100:.2f}%")
print(f"  Macro F1  (TTA)      : {report_dict['macro avg']['f1-score']*100:.2f}%")
print(f"  Weighted F1 (TTA)    : {report_dict['weighted avg']['f1-score']*100:.2f}%")
print("=" * 55)
display(summary)
